# Lab | Data Structuring and Combining Data

## Challenge 1: Combining & Cleaning Data

In this challenge, we will be working with the customer data from an insurance company, as we did in the two previous labs. The data can be found here:
- https://raw.githubusercontent.com/data-bootcamp-v4/data/main/file1.csv

But this time, we got new data, which can be found in the following 2 CSV files located at the links below.

- https://raw.githubusercontent.com/data-bootcamp-v4/data/main/file2.csv
- https://raw.githubusercontent.com/data-bootcamp-v4/data/main/file3.csv

Note that you'll need to clean and format the new data.

Observation:
- One option is to first combine the three datasets and then apply the cleaning function to the new combined dataset
- Another option would be to read the clean file you saved in the previous lab, and just clean the two new files and concatenate the three clean datasets

In [9]:
import pandas as pd

df1 = pd.read_csv("https://raw.githubusercontent.com/data-bootcamp-v4/data/main/file1.csv")
df2 = pd.read_csv("https://raw.githubusercontent.com/data-bootcamp-v4/data/main/file2.csv")
df3 = pd.read_csv("https://raw.githubusercontent.com/data-bootcamp-v4/data/main/file3.csv")

print(df1.columns)
print(df2.columns)
print(df3.columns)  

Index(['Customer', 'ST', 'GENDER', 'Education', 'Customer Lifetime Value',
       'Income', 'Monthly Premium Auto', 'Number of Open Complaints',
       'Policy Type', 'Vehicle Class', 'Total Claim Amount'],
      dtype='str')
Index(['Customer', 'ST', 'GENDER', 'Education', 'Customer Lifetime Value',
       'Income', 'Monthly Premium Auto', 'Number of Open Complaints',
       'Total Claim Amount', 'Policy Type', 'Vehicle Class'],
      dtype='str')
Index(['Customer', 'State', 'Customer Lifetime Value', 'Education', 'Gender',
       'Income', 'Monthly Premium Auto', 'Number of Open Complaints',
       'Policy Type', 'Total Claim Amount', 'Vehicle Class'],
      dtype='str')


In [10]:
def clean_column_names(df):
    df = df.copy()
    df.columns = df.columns.str.lower().str.replace(" ", "_")
    df = df.rename(columns={"st": "state"})
    return df

df1 = clean_column_names(df1)
df2 = clean_column_names(df2)
df3 = clean_column_names(df3)

In [11]:
customers = pd.concat([df1, df2, df3], axis=0, ignore_index=True)
print(customers.shape)
customers.head()

(12074, 11)


,customer,state,gender,education,customer_lifetime_value,income,monthly_premium_auto,number_of_open_complaints,policy_type,vehicle_class,total_claim_amount
0,RB50392,Washington,NaN,Master,NaN,0.0,1000.0,1/0/00,Personal Auto,Four-Door Car,2.704934
1,QZ44356,Arizona,F,Bachelor,697953.59%,0.0,94.0,1/0/00,Personal Auto,Four-Door Car,1131.464935
2,AI49188,Nevada,F,Bachelor,1288743.17%,48767.0,108.0,1/0/00,Personal Auto,Two-Door Car,566.472247
3,WW63253,California,M,Bachelor,764586.18%,0.0,106.0,1/0/00,Corporate Auto,SUV,529.881344
4,GA49547,Washington,M,High School or Below,536307.65%,36357.0,68.0,1/0/00,Personal Auto,Four-Door Car,17.269323


In [14]:
def clean_data(df):
    df = df.copy()

    df["gender"] = df["gender"].replace({"Femal": "F", "female": "F", "Male": "M"})

    df["state"] = df["state"].replace({"AZ": "Arizona", "Cali": "California", "WA": "Washington"})


    df["education"] = df["education"].replace({"Bachelors": "Bachelor"})

    df["vehicle_class"] = df["vehicle_class"].replace({"Sports Car": "Luxury", "Luxury SUV": "Luxury", "Luxury Car": "Luxury"})

    df["customer_lifetime_value"] = df["customer_lifetime_value"].astype(str).str.replace("%", "")
    df["customer_lifetime_value"] = pd.to_numeric(df["customer_lifetime_value"], errors="coerce")

    df["number_of_open_complaints"] = df["number_of_open_complaints"].apply(lambda x: x.split("/")[1] if isinstance(x, str) and "/" in x else x)
    df["number_of_open_complaints"] = pd.to_numeric(df["number_of_open_complaints"], errors="coerce")

    df = df.dropna(how="all")
    df = df.drop_duplicates()
    df["customer_lifetime_value"] = df["customer_lifetime_value"].fillna(df["customer_lifetime_value"].median())
    df = df.fillna(df.mode().iloc[0])  

    return df.reset_index(drop=True)

customers_clean = clean_data(customers)
print(customers_clean.isna().sum())
customers_clean["gender"].value_counts()

customer                     0
state                        0
gender                       0
education                    0
customer_lifetime_value      0
income                       0
monthly_premium_auto         0
number_of_open_complaints    0
policy_type                  0
vehicle_class                0
total_claim_amount           0
dtype: int64


gender
F    4726
M    4408
Name: count, dtype: int64

# Challenge 2: Structuring Data

In this challenge, we will continue to work with customer data from an insurance company, but we will use a dataset with more columns, called marketing_customer_analysis.csv, which can be found at the following link:

https://raw.githubusercontent.com/data-bootcamp-v4/data/main/marketing_customer_analysis_clean.csv

This dataset contains information such as customer demographics, policy details, vehicle information, and the customer's response to the last marketing campaign. Our goal is to explore and analyze this data by performing data cleaning, formatting, and structuring.

In [15]:
df = pd.read_csv("https://raw.githubusercontent.com/data-bootcamp-v4/data/main/marketing_customer_analysis_clean.csv")
df.head(1)

,unnamed:_0,customer,state,customer_lifetime_value,response,coverage,education,effective_to_date,employmentstatus,gender,...,number_of_policies,policy_type,policy,renew_offer_type,sales_channel,total_claim_amount,vehicle_class,vehicle_size,vehicle_type,month
0,0,DK49336,Arizona,4809.21696,No,Basic,College,2011-02-18,Employed,M,...,9,Corporate Auto,Corporate L3,Offer3,Agent,292.8,Four-Door Car,Medsize,A,2


In [16]:
revenue_channel = df.pivot_table(index="sales_channel",
                                 values="total_claim_amount",
                                 aggfunc="sum").round(2)

revenue_channel.sort_values("total_claim_amount", ascending=False)

,total_claim_amount
sales_channel,
Agent,1810226.82
Branch,1301204.00
Call Center,926600.82
Web,706600.04


In [17]:
clv_pivot = df.pivot_table(index="education",
                           columns="gender",
                           values="customer_lifetime_value",
                           aggfunc="mean").round(2)
clv_pivot

gender,F,M
education,,
Bachelor,7874.27,7703.60
College,7748.82,8052.46
Doctor,7328.51,7415.33
High School or Below,8675.22,8149.69
Master,8157.05,8168.83


In [18]:
complaints_wide = df.pivot_table(index="policy_type",
                                 columns="month",
                                 values="number_of_open_complaints",
                                 aggfunc="sum")
complaints_wide

month,1,2
policy_type,,
Corporate Auto,443.434952,385.208135
Personal Auto,1727.605722,1453.684441
Special Auto,87.074049,95.226817


In [19]:
complaints_long = complaints_wide.reset_index().melt(id_vars="policy_type",
                                                     var_name="month",
                                                     value_name="number_of_complaints")

complaints_long.sort_values("number_of_complaints", ascending=False)

,policy_type,month,number_of_complaints
1,Personal Auto,1,1727.605722
4,Personal Auto,2,1453.684441
0,Corporate Auto,1,443.434952
3,Corporate Auto,2,385.208135
5,Special Auto,2,95.226817
2,Special Auto,1,87.074049


In [20]:
complaints_wide.stack().reset_index(name="number_of_complaints")

,policy_type,month,number_of_complaints
0,Corporate Auto,1,443.434952
1,Corporate Auto,2,385.208135
2,Personal Auto,1,1727.605722
3,Personal Auto,2,1453.684441
4,Special Auto,1,87.074049
5,Special Auto,2,95.226817


1. You work at the marketing department and you want to know which sales channel brought the most sales in terms of total revenue. Using pivot, create a summary table showing the total revenue for each sales channel (branch, call center, web, and mail).
Round the total revenue to 2 decimal points.  Analyze the resulting table to draw insights.

2. Create a pivot table that shows the average customer lifetime value per gender and education level. Analyze the resulting table to draw insights.

## Bonus

You work at the customer service department and you want to know which months had the highest number of complaints by policy type category. Create a summary table showing the number of complaints by policy type and month.
Show it in a long format table.

*In data analysis, a long format table is a way of structuring data in which each observation or measurement is stored in a separate row of the table. The key characteristic of a long format table is that each column represents a single variable, and each row represents a single observation of that variable.*

*More information about long and wide format tables here: https://www.statology.org/long-vs-wide-data/*

In [ ]:
# Your code goes here